In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Machine Learning model
from sklearn.ensemble import RandomForestClassifier

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"/content/predictive_maintenance_v3.csv")


df.head()

In [ ]:
print(
    df['failure_type'].value_counts()
)

In [ ]:
df.dtypes

In [ ]:
columns = [
    'temperature_motor',
    'vibration_rms',
    'current_phase_avg',
    'pressure_level',
    'rpm'

]

for col in columns:

    median_0 = df.loc[
        df['failure_within_24h'] == 0,
        col
    ].median()

    median_1 = df.loc[
        df['failure_within_24h'] == 1,
        col
    ].median()

    df.loc[
        (df[col].isna()) &
        (df['failure_within_24h'] == 0),
        col
    ] = median_0

    df.loc[
        (df[col].isna()) &
        (df['failure_within_24h'] == 1),
        col
    ] = median_1

In [ ]:
# Filter Records with Actual Failures
df_failure = df[
    df['failure_type'] != 'none'
].copy()

In [ ]:
df_failure.drop(
    columns=[
        'timestamp',
        'machine_id',
        'failure_within_24h',
        'rul_hours',
        'estimated_repair_cost'
    ],
    inplace=True
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

enc_machine = LabelEncoder()
enc_machine.fit(df_failure['failure_type'])

print(dict(zip(
    enc_machine.classes_,
    enc_machine.transform(enc_machine.classes_)
)))

In [ ]:
# Encode Categorical Variables
from sklearn.preprocessing import LabelEncoder

machine_encoder = LabelEncoder()

df_failure['machine_type'] = (
    machine_encoder.fit_transform(
        df_failure['machine_type']
    )
)

operating_encoder = LabelEncoder()

df_failure['operating_mode'] = (
    operating_encoder.fit_transform(
        df_failure['operating_mode']
    )
)

failure_encoder = LabelEncoder()

df_failure['failure_type'] = (
    failure_encoder.fit_transform(
        df_failure['failure_type']
    )
)

In [ ]:
X = df_failure.drop(
    'failure_type',
    axis=1
)

y = df_failure['failure_type']

In [ ]:
# Split Dataset into Training and Testing Sets
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

In [ ]:
# Train Random Forest Classification Model
model_failure = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42
)

model_failure.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = model_failure.predict(
    X_test
)

In [ ]:
# Evaluate Model Performance

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

print()

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
# Save Trained Model

import joblib

joblib.dump(
    model_failure,
    'failure_type_model.pkl'
)

In [ ]:
!pip install fastapi uvicorn pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("****************")

In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8001)
print(public_url)

In [ ]:

from fastapi import FastAPI
import joblib
import pandas as pd

app1 = FastAPI()

model = joblib.load("/content/failure_type_model.pkl")

@app1.post("/predict")
def predict(data: dict):

    df = pd.DataFrame([data])

    prediction = int(model.predict(df)[0])

    return {
        "failure_type": prediction
    }
print(app1.routes)


In [ ]:

import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run():
    uvicorn.run(app1, host="0.0.0.0", port=8001)

Thread(target=run).start()


In [ ]:
import requests

payload = {
    "machine_type": 0,
    "vibration_rms": 1.24,
    "temperature_motor": 47.33,
    "current_phase_avg": 4.93,
    "pressure_level": 21.2,
    "rpm": 85.2,
    "operating_mode": 0,
    "hours_since_maintenance": 331.75,
    "ambient_temp": 8.7
}

response1 = requests.post(
    "https://YOUR-FASTAPI-ENDPOINT/predict",
    json=payload
)

print(response1.status_code)
print(response1.text)
